# 🧠 Huấn luyện AI Nhận diện Domain Quảng Cáo & Tracker (BlockAds AI)

Notebook này sử dụng máy ảo **Google Colab (GPU/High-RAM)** để:
1. Tải và xử lý dataset hàng chục nghìn tên miền (Domain Benign an toàn vs Domain Quảng cáo/Tracker).
2. Trích xuất đặc trưng Lexical & Entropy (Shannon Entropy, N-grams, tỷ lệ số, từ khóa tracking).
3. Huấn luyện mô hình Machine Learning siêu nhẹ (Gradient Boosted Trees / Random Forest).
4. Đánh giá độ chính xác (>95%) và test trực tiếp các domain mới.
5. Xuất file mô hình siêu nhẹ (ONNX / Joblib) sẵn sàng nhúng vào ứng dụng **BlockAds Android** hoặc API backend.

### Bước 1: Chuẩn bị môi trường & Thư viện

In [ ]:
# @title 1. Cài đặt các thư viện cần thiết
!pip install -q scikit-learn numpy joblib matplotlib seaborn skl2onnx onnxruntime

import os
if not os.path.exists('/content/blockads_filter_bin'):
    !git clone https://github.com/pass-with-high-score/blockads_filter_bin.git /content/blockads_filter_bin
%cd /content/blockads_filter_bin
!git pull origin main

### Bước 2: Huấn luyện mô hình AI tự động
Bạn có thể điều chỉnh số lượng mẫu (Samples) để huấn luyện.

In [ ]:
# @title 2. Chạy Pipeline Huấn luyện AI
samples_per_class = 20000 # @param {type:"integer"}

# Tự động đọc DATABASE_URL từ Secrets nếu có để lấy domain thật từ DB
import os
try:
    from google.colab import userdata
    db_url = userdata.get("DATABASE_URL")
    if db_url:
        os.environ["DATABASE_URL"] = db_url
        print("[✓] Đã kích hoạt chế độ lấy dữ liệu từ PostgreSQL Database!")
except Exception:
    pass

!python3 scripts/train_ai_model.py --samples $samples_per_class --output-dir models

### Bước 3: Thử nghiệm dự đoán trực tiếp (Interactive Tester)
Nhập bất kỳ tên miền nào bên dưới để AI phân tích ngay tức thì:

In [ ]:
# @title 3. Kiểm tra Domain với AI Model đã train
test_domain = "analytics-tracker.adservice.net" # @param {type:"string"}

import os, joblib, numpy as np, importlib
import scripts.train_ai_model
importlib.reload(scripts.train_ai_model)
from scripts.train_ai_model import extract_features

model_path = "models/adblock_domain_classifier.joblib"
if os.path.exists(model_path):
    clf = joblib.load(model_path)
    feat = np.array([extract_features(test_domain)])
    prob_ad = clf.predict_proba(feat)[0][1]
    
    print("=" * 50)
    print(f"Domain cần kiểm tra: {test_domain}")
    if prob_ad >= 0.5:
        print(f"🚫 KẾT QUẢ: CHẶN (QUẢNG CÁO / TRACKER)")
    else:
        print(f"✅ KẾT QUẢ: AN TOÀN (SAFE / BENIGN)")
    print(f"Xác suất là Quảng cáo/Tracker: {prob_ad * 100:.2f}%")
    print("=" * 50)
else:
    print("Chưa tìm thấy model. Hãy chạy Bước 2 trước!")


### Bước 4: Chuyển đổi mô hình sang ONNX (Cho Android & Mobile Apps)

In [ ]:
# @title 4. Export mô hình sang ONNX (Dung lượng siêu nhẹ < 1MB)
try:
    import joblib, os
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    
    clf = joblib.load("models/adblock_domain_classifier.joblib")
    n_feat = getattr(clf, "n_features_in_", 13)
    initial_type = [("float_input", FloatTensorType([None, n_feat]))]
    onx = convert_sklearn(clf, initial_types=initial_type)
    
    onnx_path = "models/adblock_model.onnx"
    with open(onnx_path, "wb") as f:
        f.write(onx.SerializeToString())
    print(f"[✓] Xuất thành công file ONNX: {onnx_path}")
    print(f"[✓] Số features đầu vào: {n_feat}")
    print(f"[✓] Kích thước file: {os.path.getsize(onnx_path) / 1024:.1f} KB")
except Exception as e:
    print(f"Lỗi export ONNX: {e}")


### Bước 5: Tải file Model về máy tính

In [ ]:
# @title 5. Tải file Model về
from google.colab import files
import os

for f in ["models/adblock_domain_classifier.joblib", "models/adblock_model.onnx", "models/model_meta.json"]:
    if os.path.exists(f):
        files.download(f)